In [1]:
# ==============================================================================
# CELL 1: Imports & Training Data Setup (6 Features + 1 Continuous Label)
# ==============================================================================
# Ordinary Least Squares (OLS) linear regression models a continuous target Y
# as a linear combination of predictor features X:
#     Y = beta_0 + beta_1*X_1 + ... + beta_6*X_6 + epsilon
# where epsilon is assumed to be zero-mean noise with constant variance
# (homoscedasticity).

import numpy as np
import pandas as pd
import statsmodels.api as sm

# Fixed seed for reproducible results across notebook runs
np.random.seed(42)

n_samples = 500
n_features = 6

# Generate 6 training features (feature_1 through feature_6)
feature_names = [f"feature_{i}" for i in range(1, n_features + 1)]
X_train_raw = pd.DataFrame(
    np.random.randn(n_samples, n_features), columns=feature_names
)

# Synthetic ground-truth coefficients and intercept to generate a continuous label
true_intercept = 3.0
true_weights = np.array([1.8, -1.2, 0.7, 2.4, -0.5, 1.1])
noise = np.random.normal(loc=0.0, scale=1.0, size=n_samples)

y_train = pd.Series(
    true_intercept + X_train_raw.values @ true_weights + noise,
    name="label",
)

print("X_train_raw shape:", X_train_raw.shape)
print("y_train shape    :", y_train.shape)

X_train_raw shape: (500, 6)
y_train shape    : (500,)


In [2]:
# ==============================================================================
# CELL 2: Adding Intercept & Training Linear Regression (OLS) in statsmodels
# ==============================================================================
# HOW OLS TRAINING WORKS IN STATSMODELS:
# 1. Intercept (beta_0): `statsmodels` does NOT add a constant column by
#    default. Calling `sm.add_constant()` prepends a column of 1s so the model
#    estimates an intercept term instead of forcing the line through the origin.
# 2. Objective Function: OLS minimizes the Residual Sum of Squares (RSS / SSR):
#        min_beta sum((y_i - y_hat_i)^2) = ||y - X @ beta||_2^2
# 3. Estimation Method: Under the hood, `sm.OLS.fit()` solves for the
#    parameter vector beta using linear algebra / QR or SVD decomposition
#    (equivalent to the normal equation beta_hat = (X^T X)^(-1) X^T y when
#    X^T X is invertible, while using numerically more stable matrix
#    factorizations in practice).

X_train = sm.add_constant(X_train_raw, has_constant="add")

# Instantiate the OLS linear regression model with 6 features (+1 intercept)
ols_model = sm.OLS(endog=y_train, exog=X_train)

# Fit the model on the training set
ols_results = ols_model.fit()

# Display full statistical summary (coefficients, std errors, t-stats, p-values, R^2)
print(ols_results.summary())

                            OLS Regression Results                            
Dep. Variable:                  label   R-squared:                       0.921
Model:                            OLS   Adj. R-squared:                  0.920
Method:                 Least Squares   F-statistic:                     962.8
Date:                Thu, 26 Nov 2026   Prob (F-statistic):          1.53e-268
Time:                        23:17:38   Log-Likelihood:                -718.63
No. Observations:                 500   AIC:                             1451.
Df Residuals:                     493   BIC:                             1481.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9972      0.046     64.645      0.0

In [3]:
# ==============================================================================
# CELL 3: Training Evaluation — R-Squared, Adjusted R-Squared & Error Metrics
# ==============================================================================
# HOW R-SQUARED WORKS IN OLS:
# - R-squared (Coefficient of Determination):
#       R^2 = 1 - (SS_res / SS_tot)
#   where:
#     SS_res = sum((y_i - y_hat_i)^2) is residual sum of squares (unexplained)
#     SS_tot = sum((y_i - y_bar)^2)   is total sum of squares around mean y_bar
#   It represents the proportion of variance in the training target explained
#   by the 6 features (ranging from 0 to 1 for models with an intercept).
# - Adjusted R-squared:
#   Penalizes adding extra predictors that don't improve fit enough relative to
#   degrees of freedom:
#       R^2_adj = 1 - (1 - R^2) * ((n - 1) / (n - p - 1))
#   where n = number of training samples and p = number of features (6).
# - Note: Evaluating only on training data measures in-sample fit; always
#   check a held-out validation/test set to guard against overfitting.

y_train_pred = ols_results.predict(X_train)

# Native R-squared and Adjusted R-squared from statsmodels OLSResults
r2_native = ols_results.rsquared
adj_r2_native = ols_results.rsquared_adj

# Explicit calculation for transparency / verification
ss_res = np.sum((y_train - y_train_pred) ** 2)
ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
r2_manual = 1.0 - (ss_res / ss_tot)

n = len(y_train)
p = X_train_raw.shape[1]  # 6 features
adj_r2_manual = 1.0 - (1.0 - r2_manual) * ((n - 1) / (n - p - 1))

# Complementary training error metrics
rmse = np.sqrt(np.mean((y_train - y_train_pred) ** 2))
mae = np.mean(np.abs(y_train - y_train_pred))

print(f"R-squared (ols_results.rsquared)    : {r2_native:.4f}")
print(f"R-squared (manual verification)     : {r2_manual:.4f}")
print(f"Adjusted R-squared (rsquared_adj)   : {adj_r2_native:.4f}")
print(f"Adjusted R-squared (manual)         : {adj_r2_manual:.4f}")
print(f"Training RMSE                       : {rmse:.4f}")
print(f"Training MAE                        : {mae:.4f}")

R-squared (ols_results.rsquared)    : 0.9214
R-squared (manual verification)     : 0.9214
Adjusted R-squared (rsquared_adj)   : 0.9204
Adjusted R-squared (manual)         : 0.9204
Training RMSE                       : 1.0185
Training MAE                        : 0.8256
